# Multi-level DMD

`dmd.mrDMD` on 1, 2 and 4 windows of the record. It is a windowed hierarchy: slow modes are not subtracted between levels. Amplitudes are fitted to the first snapshot of each window.

Data: GLORYS04 SSH (364 days) and surface U and V in the Gulf Stream (731 days, 30-45N, 75-45W). Animations show the anomaly beside its reconstruction.

In [ ]:
import numpy as np
from showcase import *
from modredkit import dmd, pod

RANK = 50

## SSH

In [ ]:
A, area, wet = load_ssh()
print("SSH anomalies", A.shape)
mds, eig, ct, amp, idx = dmd.mrDMD(A, 20, DT, 3)
for l in range(3):
    pick = (idx[:, 0] == l) & (np.abs(ct.imag) > 1e-8)
    per = 2*np.pi/np.abs(ct.imag[pick])
    print(f"level {l}: {2**l} windows of {A.shape[1]//2**l} days, {pick.sum()} oscillating modes, median period {np.median(per):.0f} d")
    show_spectrum(ct[idx[:, 0] == l], amp[idx[:, 0] == l], f"SSH: level {l} spectrum")

In [ ]:
s = A.shape[1] // 4
rec = np.zeros((A.shape[0], 4*s))
for j in range(4):
    pick = (idx[:, 0] == 2) & (idx[:, 1] == j)
    rec[:, j*s:(j+1)*s] = rebuild(mds[:, pick], amp[pick], eig[pick], s)
animate([A, rec], ["SSH anomaly", "Level 2, 4 windows"], wet, units="m")

## Gulf Stream U and V

In [ ]:
U, V, wet_u, wet_v = load_gulfstream()
Z = np.vstack((U, V))
nu = U.shape[0]
print("U", U.shape, "V", V.shape)
mds, eig, ct, amp, idx = dmd.mrDMD(Z, 20, DT, 3)
for l in range(3):
    pick = (idx[:, 0] == l) & (np.abs(ct.imag) > 1e-8)
    per = 2*np.pi/np.abs(ct.imag[pick])
    print(f"level {l}: {2**l} windows of {Z.shape[1]//2**l} days, {pick.sum()} oscillating modes, median period {np.median(per):.0f} d")
    show_spectrum(ct[idx[:, 0] == l], amp[idx[:, 0] == l], f"Gulf Stream: level {l} spectrum")

In [ ]:
s = Z.shape[1] // 4
rec = np.zeros((Z.shape[0], 4*s))
for j in range(4):
    pick = (idx[:, 0] == 2) & (idx[:, 1] == j)
    rec[:, j*s:(j+1)*s] = rebuild(mds[:, pick], amp[pick], eig[pick], s)
animate([U, rec[:nu]], ["U anomaly", "Level 2, 4 windows"], wet_u, step=12, units="m/s")